In [ ]:
import numpy as np
import pandas as pd
from os.path import join as pjoin
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.preprocessing import StandardScaler
from matplotlib import pyplot as plt
import copy
import random
from tqdm import tqdm

In [ ]:
root = '../data/multitask_processed'

In [ ]:
xPath = pjoin(root, 'X.h5')
yPath = pjoin(root, 'Y.h5')

In [ ]:
X = pd.read_hdf(xPath)
Y = pd.read_hdf(yPath)

In [ ]:
((X.DPHI[:200] + X.NPHI[:200])/2).plot()
Y.PHI[:200].plot()

In [ ]:
Y.PHI.describe()

In [ ]:
data = X.GR
label = X.VSH

In [ ]:
well_names = X.UWI.unique()

In [ ]:
inps, gts = [], []
step = 49
for well_name in tqdm(well_names):
    well_data = data[X.UWI == well_name]
    well_label = label[X.UWI == well_name]
    for i in range(step, well_data.shape[0]):
        inp = well_data.iloc[i-step:i+1+step]
        if inp.shape[0] == (step*2)+1:
            lbl = well_label.iloc[i+1]
            inps.append(inp.iloc[::2])
            gts.append(lbl)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(inps, gts, test_size=0.9, random_state=42)

In [ ]:
from sklearn.ensemble import RandomForestRegressor

In [ ]:
classifier = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1, verbose=1)

classifier.fit(X_train, y_train)

In [ ]:
from sklearn.metrics import mean_absolute_error, r2_score

In [ ]:
print(mean_absolute_error(y_train, classifier.predict(X_train)), r2_score(y_train, classifier.predict(X_train)))
print(mean_absolute_error(y_test, classifier.predict(X_test)), r2_score(y_test, classifier.predict(X_test)))

In [ ]:
pred = classifier.predict(inps)

In [ ]:
len(gts)

In [ ]:
print(mean_absolute_error(gts, classifier.predict(inps)), r2_score(gts, classifier.predict(inps)))